In [69]:
import tensorflow as tf

In [70]:
class GPTEmbedding(tf.keras.layers.Layer):
  def __init__(self,vocab_size,max_length,d_model):
    super().__init__()
    self.token_embedding=tf.keras.layers.Embedding(input_dim=vocab_size,output_dim=d_model)
    self.position_embedding = tf.keras.layers.Embedding(input_dim=max_length, output_dim=d_model)
  def call(self,input_ids):
    seq_length=tf.shape(input_ids)[1]
    positions=tf.range(start=0,limit=seq_length,delta=1)
    token_embeddings = self.token_embedding(input_ids)
    position_embeddings = self.position_embedding(positions)
    embeddings = token_embeddings + position_embeddings
    return embeddings


In [71]:
class MultiHeadAttention(tf.keras.layers.Layer):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    assert d_model % num_heads == 0
    self.head_dim = d_model // num_heads
    self.query=tf.keras.layers.Dense(d_model)
    self.keys=tf.keras.layers.Dense(d_model)
    self.values=tf.keras.layers.Dense(d_model)
    self.output_projection = tf.keras.layers.Dense(d_model)
    self.scale=tf.math.sqrt(tf.cast(self.head_dim, tf.float32))
  def split_heads(self,x):
    batch_size=tf.shape(x)[0]
    seq_length=tf.shape(x)[1]
    x=tf.reshape(x,(batch_size,seq_length,self.num_heads,self.head_dim))
    x=tf.transpose(x,perm=[0,2,1,3])
    return x
  def call(self, inputs, attention_mask=None):

    Q = self.query(inputs)
    K = self.keys(inputs)
    V = self.values(inputs)

    Q = self.split_heads(Q)
    K = self.split_heads(K)
    V = self.split_heads(V)

    scores = tf.matmul(Q, K, transpose_b=True)
    scaled_scores = scores / self.scale

    seq_length = tf.shape(inputs)[1]

    # Causal mask
    causal_mask = tf.linalg.band_part(
        tf.ones((seq_length, seq_length)),
        -1, 0
    )

    # Expand dimensions
    causal_mask = causal_mask[tf.newaxis, tf.newaxis, :, :]

    # Combine with padding mask
    if attention_mask is not None:
        padding_mask = attention_mask[:, tf.newaxis, tf.newaxis, :]

        combined_mask = tf.logical_and(
            tf.cast(causal_mask, tf.bool),
            tf.cast(padding_mask, tf.bool)
        )
    else:
        combined_mask = tf.cast(causal_mask, tf.bool)

    masked_scores = tf.where(
        combined_mask,
        scaled_scores,
        -1e9
    )

    attention_weights = tf.nn.softmax(masked_scores, axis=-1)

    output = tf.matmul(attention_weights, V)
    output = tf.transpose(output, perm=[0, 2, 1, 3])

    batch_size = tf.shape(inputs)[0]

    output = tf.reshape(
        output,
        (batch_size, seq_length, self.d_model)
    )

    output = self.output_projection(output)

    return output

In [72]:
attention = MultiHeadAttention(d_model=256, num_heads=8)

x = tf.random.normal((1, 5, 256))

output = attention(x)

print("Test 1")
print("Input shape:", x.shape)
print("Output shape:", output.shape)


x = tf.random.normal((2, 10, 256))

output = attention(x)

print("\nTest 2")
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Test 1
Input shape: (1, 5, 256)
Output shape: (1, 5, 256)

Test 2
Input shape: (2, 10, 256)
Output shape: (2, 10, 256)


In [73]:
class FeedForwardNetwork(tf.keras.layers.Layer):
  def __init__(self,d_model,ffn):
    super().__init__()
    self.dense1=tf.keras.layers.Dense(ffn,activation="gelu")
    self.dense2=tf.keras.layers.Dense(d_model)
  def call(self,inputs):
    x=self.dense1(inputs)
    x=self.dense2(x)
    return x


In [74]:
ffn = FeedForwardNetwork(256,1024)

x = tf.random.normal((1, 5, 256))

output = ffn(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: (1, 5, 256)
Output shape: (1, 5, 256)


In [75]:
class GPTTransformerBlock(tf.keras.layers.Layer):
  def __init__(self,d_model,num_heads,dff,dropout_rate=0.1):
    super().__init__()
    self.attention=MultiHeadAttention(d_model, num_heads)
    self.ffn=FeedForwardNetwork(d_model,dff)
    self.layer_norm1=tf.keras.layers.LayerNormalization(epsilon=1e-5)
    self.layer_norm2=tf.keras.layers.LayerNormalization(epsilon=1e-5)
    self.dropout1 = tf.keras.layers.Dropout(dropout_rate)
    self.dropout2 = tf.keras.layers.Dropout(dropout_rate)
  def call(self, inputs, training=False, attention_mask=None):
    x = self.layer_norm1(inputs)
    attention_output = self.attention(
    x,
    attention_mask=attention_mask
)
    attention_output = self.dropout1(attention_output, training=training)
    x = inputs + attention_output
    normalized_x = self.layer_norm2(x)
    ffn_output = self.ffn(normalized_x)
    ffn_output = self.dropout2(ffn_output, training=training)
    output = x + ffn_output
    return output

In [76]:
transformer_block = GPTTransformerBlock(
    d_model=256,
    num_heads=8,
    dff=1024
)

x = tf.random.normal((1, 5, 256))

output = transformer_block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: (1, 5, 256)
Output shape: (1, 5, 256)


In [81]:
class GPTDecoder(tf.keras.Model):
  def __init__(self,vocab_size,num_heads,max_length,d_model,dff,num_layers,dropout_rate=0.1):
    super().__init__()
    self.embedding=GPTEmbedding(vocab_size=vocab_size,max_length=max_length,d_model=d_model)
    self.transformer_block=[GPTTransformerBlock(d_model=d_model,num_heads=num_heads,dff=dff,dropout_rate=dropout_rate)for _ in range(num_layers)]
    self.final_layer_norm = tf.keras.layers.LayerNormalization(epsilon=1e-5 )
    self.lm_head = tf.keras.layers.Dense(vocab_size)
  def call(self, input_ids, training=False, attention_mask=None):
    x=self.embedding(input_ids)
    for block in self.transformer_block:
      x = block(
    x,
    training=training,
    attention_mask=attention_mask
)
    x = self.final_layer_norm(x)
    logits = self.lm_head(x)
    return logits






In [78]:
VOCAB_SIZE = 10000
MAX_LENGTH = 512
D_MODEL = 256
NUM_HEADS = 8
DFF = 1024
NUM_LAYERS = 4
DROPOUT_RATE = 0.1

In [87]:
gpt = GPTDecoder(
    vocab_size=VOCAB_SIZE,
    num_heads=NUM_HEADS,
    max_length=MAX_LENGTH,
    d_model=D_MODEL,
    dff=DFF,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE
)



In [88]:
input_ids = tf.random.uniform(
    shape=(2, 10),
    minval=0,
    maxval=VOCAB_SIZE,
    dtype=tf.int32
)

logits = gpt(input_ids, training=False)

print("Input shape:", input_ids.shape)
print("Output shape:", logits.shape)
print("Total parameters:", gpt.count_params())

Input shape: (2, 10)
Output shape: (2, 10, 10000)
Total parameters: 8420624


In [86]:
import inspect

print(inspect.signature(GPTDecoder.call))

(self, input_ids, training=False, attention_mask=None)


In [89]:
attention_mask = tf.constant([
    [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
], dtype=tf.int32)

logits = gpt(
    input_ids,
    training=False,
    attention_mask=attention_mask
)

print("Input shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Output shape:", logits.shape)

Input shape: (2, 10)
Attention mask shape: (2, 10)
Output shape: (2, 10, 10000)
